This script is an experiment in analyzing multi-hazard impacts on the power grid

The goal of this script is to identify a candidate set of wildfire + space weather events

Approach (implemented)
- Get wildfire point 
- Choose radius of some sort
- Get any power grid disturbance that occurred within that time and circle
- Inspect space weather data for preceding day to current time
- Store full multi-hazard data for the candidate event and study forensically

Possibility of a different approach (not implemented):
- Read in set of wildfire events from MODIS or Suomi
- Get time period, get spatial extent (GIS) -> visualize it
- Highest level: get space weather indices
    - more granular is to get magnetometers and GNSS data (GIMs, likely) in the area of the fire 
- Get HIFLD grid affected based on the wildfire
- (optional) get observed power grid disturbances that overlap in space and time
- Record the candidate event for forensic analysis (social media, interviews, grey literature exploration)


TODO
- the connection may very well come from space weather effects on communications during wildfire events (need to incorporate radiation data to represent space weather) 
- explore emdat database
- add sophistication to wildfire region processing
- add terrestrial weather to search
- explore 2024 data

#### Dependencies and Custom Functions

In [ ]:
import numpy as np
import pandas as pd
import os, sys

from datetime import datetime, time
import geopandas as gpd

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.dates import DateFormatter

from shapely.geometry import Polygon, MultiPolygon

# import contextily as ctx


from supermag_api.supermag_api import *


import re


In [ ]:
def get_sm_one_minute(start,end):

    '''

    
    Dependencies
        supermag-api downloadable from https://supermag.jhuapl.edu/mag/?fidelity=low&start=2001-01-01T00%3A00%3A00.000Z&interval=1%3A00%3A00&tab=api
            --> one should move this to the working directory and rename to supermag_api
        datetime
            
    Notes
        it appears this will only work for a few-day request; for longer periods, need to download the data directly
            
    '''


#     import supermag_api
#     import datetime

    start_datetime = pd.to_datetime(start)
    end_datetime = pd.to_datetime(end)
    
    # get the number of days between start and end
    num_days = (end_datetime - start_datetime).days
    print('the number of days that will be requested is {}'.format(num_days))
    input('Press Enter if you are sure you want to ping the SuperMAG API for this...')
    
    # get correct format for 'start date'
    # date_obj_start = datetime.strptime(start, '%Y-%m-%d %H:%M:%S')
    # formatted_start = date_obj_start.strftime('%Y-%m-%dT%H:%M')
    date_obj_start = datetime.datetime.strptime(start, '%Y-%m-%d %H:%M:%S')
    formatted_start = date_obj_start.strftime('%Y-%m-%dT%H:%M')

    (status,df_smidxs) = SuperMAGGetIndices('rymc1012',formatted_start,86400*num_days,'SME, SMU, SML, SMR')

    datetimes_sm = [pd.to_datetime(t, unit='s') for t in df_smidxs['tval']]

    df_smidxs['datetimes'] = datetimes_sm
    df_smidxs = df_smidxs.set_index(['datetimes'])
    df_smidxs.index = pd.to_datetime(df_smidxs.index)
    
    return df_smidxs





#### Wildfire database

Some things I'm learning about wildfire data
- MODIS might be best available given coverage
- There is a NASA digital twin project https://ideas-digitaltwin.jpl.nasa.gov/airquality/, but it is focused on air quality and thus the atmospheric emissions associated with wildfire, whereas for this work I need the wildfires themselves (not their emissions which then mix with the atmosphere and its dynamics)
- ...


In [ ]:
# NASA FIRMS data https://firms.modaps.eosdis.nasa.gov


# National Interagency Fire Center https://www.nifc.gov/
file_path_nifc_wildfire = "/Users/ryanmc/Documents/Conferences/Jack_Eddy_Symposium_2022/dev/wildfire_data/WFIGS_Incident_Locations_6818922478522849878.geojson"
file_path_modis_wildfire = "/Users/ryanmc/Documents/Conferences/Jack_Eddy_Symposium_2022/dev/wildfire_data/DL_FIRE_M-C61_530609/fire_archive_M-C61_530609.shp"
# file_path_suomi_wildfire = "/Users/ryanmc/Documents/Conferences/Jack_Eddy_Symposium_2022/dev/wildfire_data/DL_FIRE_SV-C2_530613/fire_archive_SV-C2_530613.shp"

# gdf_nifc_wildfire = gpd.read_file(file_path_nifc_wildfire)
gdf_modis_wildfire = gpd.read_file(file_path_modis_wildfire)
# gdf_suomi_wildfire = gpd.read_file(file_path_suomi_wildfire)




In [ ]:
print('size of the wildfire data is: {:.2f} MB'.format(sys.getsizeof(gdf_modis_wildfire) / 10**6))

In [ ]:
# Explore the first few rows
print(gdf_modis_wildfire.head())

# Check the geometry type (e.g., Point, Polygon, LineString)
print(gdf_modis_wildfire.geom_type.unique())

# Get summary of the attributes
print(gdf_modis_wildfire.info())

# # Plot the shapefile (quick visualization)
# gdf.plot()


In [ ]:
if 'ACQ_DATE' in gdf_modis_wildfire.columns and 'ACQ_TIME' in gdf_modis_wildfire.columns:
    gdf_modis_wildfire['datetime'] = (
        gdf_modis_wildfire['ACQ_DATE'].astype(str) + ' ' + gdf_modis_wildfire['ACQ_TIME'].astype(str)
    )
    gdf_modis_wildfire['datetime'] = pd.to_datetime(gdf_modis_wildfire['datetime'])
else:
    raise ValueError("Expected 'ACQ_DATE' and 'ACQ_TIME' columns.")

### (optional) visualize the wildfire data

In [ ]:
import folium
from folium.plugins import TimestampedGeoJson



In [ ]:



# Convert GeoDataFrame to GeoJSON-like dictionary with timestamp for Folium
features = []
for _, row in gdf_modis_wildfire.iterrows():
    feature = {
        'type': 'Feature',
        'geometry': row['geometry'].__geo_interface__,
        'properties': {
            'time': row['datetime'].isoformat(),  # Use ISO format for timestamps
            'popup': f"Acquired: {row['datetime']}"
        }
    }
    features.append(feature)

# Create the Timestamped GeoJSON layer
timestamped_geojson = TimestampedGeoJson(
    {'type': 'FeatureCollection', 'features': features},
    period='PT1H',  # Period for animation: 1 hour
    add_last_point=False,
    auto_play=False,
    loop=True,
    max_speed=1,
    loop_button=True,
    date_options='YYYY-MM-DD HH:mm:ss',
    time_slider_drag_update=True
)

# Initialize a Folium map centered on the dataset's centroid
m = folium.Map(
    location=[gdf_modis_wildfire.geometry.y.mean(), gdf_modis_wildfire.geometry.x.mean()],
    zoom_start=6, tiles='CartoDB positron'
)

# Add the timestamped layer to the map
timestamped_geojson.add_to(m)

# Save the map as an HTML file and display it
m.save('interactive_map.html')

# If running in Jupyter, display the map directly
m


### Create a list of features grouped by unique timestamps


In [ ]:
features = []
for dt, group in gdf_modis_wildfire.groupby('datetime'):
    feature = {
        'type': 'FeatureCollection',
        'features': [
            {
                'type': 'Feature',
                'geometry': row['geometry'].__geo_interface__,
                'properties': {
                    'time': dt.isoformat(),  # Set timestamp in ISO format
                    'popup': f"Acquired: {dt}"
                }
            }
            for _, row in group.iterrows()
        ]
    }
    features.append(feature)


# # Create the TimestampedGeoJson with these features
# timestamped_geojson = TimestampedGeoJson(
#     {'type': 'FeatureCollection', 'features': features},
#     period='PT1H',              # Set period of time for each step (1 hour)
#     add_last_point=False,        # Ensure only current points appear
#     auto_play=False,              # Automatically start animation
#     loop=True,                   # Loop the animation
#     max_speed=1,                 # Normal animation speed
#     loop_button=True,            # Allow user to loop animation
#     date_options='YYYY-MM-DD HH:mm:ss',  # Display format for timestamps
#     time_slider_drag_update=True # Update on dragging time slider
# )

# # Initialize the Folium map centered on the dataset's centroid
# m = folium.Map(
#     location=[gdf_modis_wildfire.geometry.y.mean(), gdf_modis_wildfire.geometry.x.mean()],
#     zoom_start=6, 
#     tiles='CartoDB positron'  # Light basemap
# )

# # Add the timestamped GeoJSON to the map
# timestamped_geojson.add_to(m)

# # Save the map as an HTML file and display it
# m.save('interactive_map.html')

# # If running in Jupyter, display the map directly
# m

#### Optionally read in the HIFLD data for power grid transmission lines
uses HIFLD data: https://hifld-geoplatform.hub.arcgis.com/datasets/geoplatform::transmission-lines

In [ ]:
file_path_powergrid = "/Users/ryanmc/Documents/Conferences/Jack_Eddy_Symposium_2022/dev/physical_grid_data/U.S._Electric_Power_Transmission_Lines.geojson"
gdf_powergrid = gpd.read_file(file_path_powergrid)



In [ ]:
print('size of the power grid data is: {:.2f} MB'.format(sys.getsizeof(gdf_powergrid) / 10**6))

#### Power Grid Disturbance Data

Candidates
- [new dataset from EAGLE-I](https://www.nature.com/articles/s41597-024-03095-5) (Data are available in the Figshare repository at https://doi.org/10.6084/m9.figshare.24237376 or via archive request from the eagle-I website)
- [DOE electric disturbance events OE-417](https://www.oe.netl.doe.gov/OE417_annual_summary.aspx)


New Candidate: [NASA Black Marble](https://blackmarble.gsfc.nasa.gov/) (will require more thinking about how we might use these data)


In [ ]:
# Read in 2023 data
outage_directory = '/Users/ryanmc/Documents/Conferences/Jack_Eddy_Symposium_2022/dev/outage_data/'
oe417 = pd.read_excel(os.path.join(outage_directory,'DOE-OE-417/2023_Annual_Summary.xlsx'),header=1)
oe417 = oe417.dropna(subset=['Time Event Began'])
oe417 = oe417.iloc[:-2]
for o in range(len(oe417['Number of Customers Affected'])):
    if oe417['Number of Customers Affected'].iloc[o]=='Unknown':
        oe417['Number of Customers Affected'].iloc[o] = np.nan
    elif type(oe417['Number of Customers Affected'].iloc[o]) == str: 
        oe417['Number of Customers Affected'].iloc[o] = int(oe417['Number of Customers Affected'].iloc[o])
    
oe417 = oe417[oe417['Number of Customers Affected']>1000]

eaglei = pd.read_csv(os.path.join(outage_directory,'EAGLE-I/eaglei_outages_2023.csv'))
# limit the data to >1000 customers out
eaglei = eaglei[eaglei['customers_out']>1000]


In [ ]:
print('size of the oe417 data is: {:.2f} MB'.format(sys.getsizeof(oe417) / 10**6))
print('size of the eaglei data is: {:.2f} MB'.format(sys.getsizeof(eaglei) / 10**6))

In [ ]:
oe417

In [ ]:

from dateutil.parser import parse

def get_datetime_oe417(date,time):
    # try: 
    #     dt = datetime.combine(str_date,dt_time)
    # except: 
    #     try:
    #         date_obj = datetime.strptime(str_date, '%m/%d/%Y')
    #         # dt = datetime.combine(date_obj, dt_time)
    #         dt = parse(date_obj+' '+dt_time)
    #     except:
    #         dt = np.nan

    if type(date) == str:
        date_obj = datetime.strptime(date, '%m/%d/%Y')
        dt = datetime.combine(date_obj,time)
    else:
        dt = datetime.combine(date,time)
    
    return dt

# Regular expression to capture 'State: County' pairs
pattern = r'(\w+):\s*([\w\s-]+)'

# Function to extract states and counties from each row
# def extract_state_county_pairs(row):
#     pairs = []
#     # Split by semicolon to handle multiple state-county pairs
#     entries = row.split(';')
#     for entry in entries:
#         # Use regex to extract state and county
#         match = re.match(r'(\w+):\s*(.*)', entry.strip())
#         if match:
#             state = match.group(1)
#             county = match.group(2)
#             pairs.append((state, county))
#     return pairs

def extract_state_county_pairs(location_str):
    pairs = []

    # Split by ';' to handle multiple state-county blocks
    state_county_blocks = location_str.split(';')

    for block in state_county_blocks:
        block = block.strip()

        # Case: Handle multiple consecutive states without counties (e.g., "Alabama: Maine:")
        if re.fullmatch(r"(.+?:)+", block):
            states = [state.strip() for state in block.split(':') if state.strip()]
            for state in states:
                pairs.append((state, None))  # Store each state with None for county
            continue  # Skip further processing for this block

        # Use regex to match "State: County, County, ..." or "State:"
        match = re.match(r"(.+?):\s*(.*)", block)

        if match:
            state = match.group(1).strip()
            counties = match.group(2).strip()

            if counties:  # If counties exist, split them by ','
                counties_list = [county.strip() for county in counties.split(',')]
                for county in counties_list:
                    pairs.append((state, county))
            else:  # If no counties are provided
                pairs.append((state, None))
        else:
            # If the block contains only a state name
            pairs.append((block, None))

    return pairs


def get_oe417_geometries(row_area_affected,gdf_states,gdf_counties):
    '''
        inputs: 
            - row of oe417 data for Area Affected (oe417['Area Affected'].iloc[x])
            - data for states geometries
            - data for county geometries
        dependencies:
            - extract_state_county_pairs function
    '''
    
    row_states_counties = extract_state_county_pairs(row_area_affected)
    all_geometries = []
    for l in row_states_counties: 
        if not l[0]:
            # skip the empty rows in the extracted data
            continue
        else:
            if not l[1]:
                # if there is no county data, then just use whole state
                geom_data_tmp = gpd.GeoSeries(gdf_states['geometry'][gdf_states['NAME'] == l[0]].values)
                if geom_data_tmp.empty:
                    continue
                if geom_data_tmp.apply(lambda geom: isinstance(geom, MultiPolygon)).all():
#                     print('----> encountering multipolygon for {}, skipping'.format(l[0]))
                    all_geometries.extend([polygon for polygon in geom_data_tmp.geometry[0].geoms])
                else:
                    all_geometries.extend( [Polygon(geom_data_tmp.geometry[0])] )
            else:
                # if there is county data, get the county geometries and combine them
                fips_tmp = get_fips(l[0])
                geom_data_tmp = gpd.GeoSeries(gdf_counties['geometry'][ (gdf_counties['STATEFP'] == fips_tmp) & (gdf_counties['NAMELSAD'] == l[1]) ].values)
                if geom_data_tmp.empty:
                    continue
                if geom_data_tmp.apply(lambda geom: isinstance(geom, MultiPolygon)).all():
#                     print('----> encountering multipolygon for {}, skipping'.format(l[0]))
                    all_geometries.extend([polygon for polygon in geom_data_tmp.geometry[0].geoms])
                else:
                    all_geometries.extend( [Polygon(geom_data_tmp.geometry[0])] )

    combined_geometry = gpd.GeoSeries(all_geometries).unary_union

    return combined_geometry, row_states_counties

def get_eaglei_geometries(row_county,row_state,gdf_states,gdf_counties):
    '''
        inputs: 
            - row of eaglei data for county (eagelei['county'].iloc[x])
            - row of eaglei data for state (eagelei['state'].iloc[x])
            - data for states geometries
            - data for county geometries
        dependencies:
            - extract_state_county_pairs function
    '''
    
    all_geometries = []
    fips_tmp = get_fips(row_state)
    geom_data_tmp = gpd.GeoSeries(gdf_counties['geometry'][ (gdf_counties['STATEFP'] == fips_tmp) & (gdf_counties['NAMELSAD'] == row_county + ' County') ].values)
    if geom_data_tmp.empty:
        return np.nan
    if geom_data_tmp.apply(lambda geom: isinstance(geom, MultiPolygon)).all():
        all_geometries.extend([polygon for polygon in geom_data_tmp.geometry[0].geoms])
    else:
        all_geometries.extend( [Polygon(geom_data_tmp.geometry[0])] )
    

    combined_geometry = gpd.GeoSeries(all_geometries).unary_union

    return combined_geometry

# Dictionary mapping state abbreviations to FIPS codes
# List of tuples containing (state_abbreviation, fips_code, full_state_name)
state_fips = [
    ('AL', '01', 'Alabama'),
    ('AK', '02', 'Alaska'),
    ('AZ', '04', 'Arizona'),
    ('AR', '05', 'Arkansas'),
    ('CA', '06', 'California'),
    ('CO', '08', 'Colorado'),
    ('CT', '09', 'Connecticut'),
    ('DE', '10', 'Delaware'),
    ('FL', '12', 'Florida'),
    ('GA', '13', 'Georgia'),
    ('HI', '15', 'Hawaii'),
    ('ID', '16', 'Idaho'),
    ('IL', '17', 'Illinois'),
    ('IN', '18', 'Indiana'),
    ('IA', '19', 'Iowa'),
    ('KS', '20', 'Kansas'),
    ('KY', '21', 'Kentucky'),
    ('LA', '22', 'Louisiana'),
    ('ME', '23', 'Maine'),
    ('MD', '24', 'Maryland'),
    ('MA', '25', 'Massachusetts'),
    ('MI', '26', 'Michigan'),
    ('MN', '27', 'Minnesota'),
    ('MS', '28', 'Mississippi'),
    ('MO', '29', 'Missouri'),
    ('MT', '30', 'Montana'),
    ('NE', '31', 'Nebraska'),
    ('NV', '32', 'Nevada'),
    ('NH', '33', 'New Hampshire'),
    ('NJ', '34', 'New Jersey'),
    ('NM', '35', 'New Mexico'),
    ('NY', '36', 'New York'),
    ('NC', '37', 'North Carolina'),
    ('ND', '38', 'North Dakota'),
    ('OH', '39', 'Ohio'),
    ('OK', '40', 'Oklahoma'),
    ('OR', '41', 'Oregon'),
    ('PA', '42', 'Pennsylvania'),
    ('RI', '44', 'Rhode Island'),
    ('SC', '45', 'South Carolina'),
    ('SD', '46', 'South Dakota'),
    ('TN', '47', 'Tennessee'),
    ('TX', '48', 'Texas'),
    ('UT', '49', 'Utah'),
    ('VT', '50', 'Vermont'),
    ('VA', '51', 'Virginia'),
    ('WA', '53', 'Washington'),
    ('WV', '54', 'West Virginia'),
    ('WI', '55', 'Wisconsin'),
    ('WY', '56', 'Wyoming'),
]

def get_fips(state_name):
    for abbr, fips, name in state_fips:
        if name.upper() == state_name.upper():
            return fips
    return None, None  # Return None if not found

##### Country/State location data
Geting the data from [US Census Bureau Tiger platform](https://www.census.gov/geographies/mapping-files/time-series/geo/tiger-line-file.html)

In [8]:
gdf_states = gpd.read_file('/Users/ryanmc/Documents/Conferences/Jack_Eddy_Symposium_2022/dev/location_data/Census_Bureau_Data/tl_2023_us_state/tl_2023_us_state.shp')
gdf_counties = gpd.read_file('/Users/ryanmc/Documents/Conferences/Jack_Eddy_Symposium_2022/dev/location_data/Census_Bureau_Data/tl_2023_us_county/tl_2023_us_county.shp')

In [ ]:
print('size of the states data is: {:.2f} MB'.format(sys.getsizeof(gdf_states) / 10**6))
print('size of the counties data is: {:.2f} MB'.format(sys.getsizeof(gdf_counties) / 10**6))


In [ ]:
# Merge into one outage data file as a gpd
outage_columns = ['outage_start_time', 'outage_stop_time', 'customers_affected','geometry','source']
outage_data_gdf = gpd.GeoDataFrame(pd.DataFrame(columns=outage_columns), geometry='geometry')

outage_data_gdf.set_crs("EPSG:4326", inplace=True)  # WGS84 - lat/lon

In [ ]:
# growing the outage data with oe417 data

for o in range(len(oe417)):
    
    # get start and end times
    str_date_start = oe417['Date Event Began'].iloc[o]
    dt_time_start = oe417['Time Event Began'].iloc[o]
    str_date_end = oe417['Date of Restoration'].iloc[o]
    dt_time_end = oe417['Time of Restoration'].iloc[o]
    try:
        dt_start = get_datetime_oe417(str_date_start,dt_time_start)
    except:
        dt_start = np.nan
    try:
        dt_end = get_datetime_oe417(str_date_end,dt_time_end)
    except:
        dt_end = np.nan    

    # get the geometries of the areas affected
    combined_geometry, states_counties = get_oe417_geometries(oe417['Area Affected'].iloc[o],gdf_states,gdf_counties)
    
    new_row = {
        'source': 'oe417', 
        'area affected': states_counties, 
        'outage_start_time': dt_start,
        'outage_stop_time': dt_end,
        'customers_affected': oe417['Number of Customers Affected'].iloc[o],
        'geometry': combined_geometry
    }
    
    outage_data_gdf = pd.concat([outage_data_gdf, pd.DataFrame([new_row])], ignore_index=True)

In [ ]:
outage_data_gdf#['outage_start_time'].isna().sum()

In [ ]:
# growing the outage data with eaglei data

for e in range(len(eaglei)):
    
    # get start times
    try:
        dt_start = pd.to_datetime(eaglei['run_start_time'].iloc[e], format='%m/%d/%y %H:%M')
    except:
        dt_start = np.nan
    
    combined_geometry = get_eaglei_geometries(eaglei['county'].iloc[e],eaglei['state'].iloc[e],gdf_states,gdf_counties)
    
    outage_data_gdf = outage_data_gdf.append({
                                                'source': 'eaglei', 
                                                'area affected': [(eaglei['state'].iloc[e],eaglei['county'].iloc[e]+' County')], 
                                                'outage_start_time': dt_start,
                                                'outage_stop_time': np.nan,
                                                'customers_affected': eaglei['customers_out'].iloc[e],
                                                'geometry': combined_geometry
                                                    }, ignore_index=True)

#### Candidate search


#### first group the modis wildfire data by datetime and nearby lat-long points


In [ ]:
import geopandas as gpd
from shapely.geometry import Point, MultiPoint

def aggregate_points_within_radius(gdf, radius=0.01):
    """
    Aggregates points by datetime and spatial proximity into polygons.

    Parameters:
    - df: GeoDataFrame with columns ["datetime", "geometry"]
    - radius: Proximity radius in degrees (~0.01 degrees ≈ 1.1 km) for grouping
    
    Returns:
    - A new GeoDataFrame with datetime and polygon geometry
    """
    # Group by datetime
    grouped = gdf.groupby('datetime')

    result_rows = []

    for datetime_value, group in grouped:
        points = group['geometry'].tolist()
        clustered_points = []

        # Track visited points
        visited = set()

        for i, point in enumerate(points):
            if i in visited:
                continue

            # Start a cluster
            cluster = [point]
            visited.add(i)

            for j, other_point in enumerate(points):
                if j not in visited and point.distance(other_point) <= radius:
                    cluster.append(other_point)
                    visited.add(j)

            # Create a polygon from the clustered points
            polygon = MultiPoint(cluster).convex_hull
            result_rows.append({
                'datetime': datetime_value,
                'geometry': polygon
            })

    # Create a new GeoDataFrame from the result
    result_gdf = gpd.GeoDataFrame(result_rows, columns=['datetime', 'geometry'], crs=gdf.crs)
    return result_gdf


# Example usage with your GeoDataFrame
gdf_modis_wildfire_result = aggregate_points_within_radius(gdf_modis_wildfire, radius=0.01)
print(gdf_modis_wildfire_result)



##### Optionally plot grouped wildfire instances in MODIS 


In [ ]:
radius = 0.1  # Example radius in degrees (adjust based on your CRS)

candidates = []
# candidate_columns = ['outage_start_time', 'outage_stop_time', 'customers_affected','geometry','source']
# candidate_gdf = gpd.GeoDataFrame(pd.DataFrame(columns=outage_columns), geometry='geometry')


# for w in range(len(gdf_modis_wildfire)):
for index_w, row_w in gdf_modis_wildfire_result.iterrows():
    
    # Identify overlap in time with the power grid disturbance
    wildfire_datetime = row_w.datetime #gdf_modis_wildfire['datetime'].iloc[w]
    filtered_outage_data_gdf = outage_data_gdf[
                                            (outage_data_gdf['outage_start_time'] >= wildfire_datetime - pd.Timedelta(hours=1)) &
                                            (outage_data_gdf['outage_start_time'] <= wildfire_datetime + pd.Timedelta(days=1))
                                            ]
    
    # Identify overlap in space with the power grid disturbance
    geom = row_w.geometry
    
    # Check if the geometry is a Point
    if geom.geom_type == 'Point':
        #   Create a buffer (circular area) around the point
        wildfire_point = gdf_modis_wildfire['geometry'].iloc[row_w.name]
        buffered_wildfire_area = geom.buffer(radius)
        disp_geom = geom.coords[0]

    # Check if the geometry is a Polygon
    elif geom.geom_type == 'Polygon':
        buffered_wildfire_area = geom
        disp_geom = geom.exterior.coords[0]

    
    for f in range(len(filtered_outage_data_gdf)):
        if buffered_wildfire_area.intersects(filtered_outage_data_gdf['geometry'].iloc[f]):
            
            # Check space weather data for space weather conditions (if at any time in the surrounding 24 hours AE exceeded 200 nT)
            sm_data = get_sm_one_minute((wildfire_datetime - pd.Timedelta(hours=24)).strftime('%Y-%m-%d %H:%M:%S'),
                                        (wildfire_datetime + pd.Timedelta(hours=12)).strftime('%Y-%m-%d %H:%M:%S'))
            
            if sm_data['SME'][sm_data['SME']>200.].any():


                # save as a candidate
                print('candidate at {}! \n wildfire: \n \t {} \n outage: \n \t {} for {} \n sp wx: \n \t {} at {}'.format(wildfire_datetime,
                                                                                                                             disp_geom,
                                                                                                                             filtered_outage_data_gdf['outage_start_time'].iloc[f],
                                                                                                                             filtered_outage_data_gdf['area affected'].iloc[f],
                                                                                                                             sm_data['SME'][sm_data['SME']==sm_data['SME'].max()].values,
                                                                                                                             sm_data.index[sm_data['SME']==sm_data['SME'].max()].values,))
                candidates.append({
                                    'wildfire datetime': wildfire_datetime,
                                    'wildfire geometry': geom,
                                    'outage_start_time': filtered_outage_data_gdf['outage_start_time'].iloc[f],
                                    'outage_stop_time': filtered_outage_data_gdf['outage_stop_time'].iloc[f],
                                    'customers affected': filtered_outage_data_gdf['customers_affected'].iloc[f],
                                    'outage geometry': filtered_outage_data_gdf['geometry'].iloc[f],
                                    'area affected': filtered_outage_data_gdf['area affected'].iloc[f],
                                    'max SME': sm_data['SME'][sm_data['SME']==sm_data['SME'].max()].values,

                                    })
                break
    continue

#### Candidate search

In [ ]:

radius = 0.1  # Example radius in degrees (adjust based on your CRS)

candidates = []
# candidate_columns = ['outage_start_time', 'outage_stop_time', 'customers_affected','geometry','source']
# candidate_gdf = gpd.GeoDataFrame(pd.DataFrame(columns=outage_columns), geometry='geometry')


# for w in range(len(gdf_modis_wildfire)):
for index_w, row_w in gdf_modis_wildfire_result.iterrows():
    
    # Identify overlap in time with the power grid disturbance
    wildfire_datetime = row_w.datetime #gdf_modis_wildfire['datetime'].iloc[w]
    filtered_outage_data_gdf = outage_data_gdf[
                                            (outage_data_gdf['outage_start_time'] >= wildfire_datetime - pd.Timedelta(hours=1)) &
                                            (outage_data_gdf['outage_start_time'] <= wildfire_datetime + pd.Timedelta(days=1))
                                            ]
    
    # Identify overlap in space with the power grid disturbance
    geom = row_w.geometry
    
    # Check if the geometry is a Point
    if geom.geom_type == 'Point':
        #   Create a buffer (circular area) around the point
        wildfire_point = gdf_modis_wildfire['geometry'].iloc[row_w.name]
        buffered_wildfire_area = geom.buffer(radius)
        disp_geom = geom.coords[0]

    # Check if the geometry is a Polygon
    elif geom.geom_type == 'Polygon':
        buffered_wildfire_area = geom
        disp_geom = geom.exterior.coords[0]

    
    for f in range(len(filtered_outage_data_gdf)):
        if buffered_wildfire_area.intersects(filtered_outage_data_gdf['geometry'].iloc[f]):
            
            # Check space weather data for space weather conditions (if at any time in the surrounding 24 hours AE exceeded 200 nT)
            sm_data = get_sm_one_minute((wildfire_datetime - pd.Timedelta(hours=24)).strftime('%Y-%-m-%-d %H:%M:%S'),
                                        (wildfire_datetime + pd.Timedelta(hours=12)).strftime('%Y-%-m-%-d %H:%M:%S'))
            
            if sm_data['SME'][sm_data['SME']>200.].any():


                # save as a candidate
                print('candidate at {}! \n wildfire: \n \t {} \n outage: \n \t {} for {} \n sp wx: \n \t {} at {}'.format(wildfire_datetime,
                                                                                                                             disp_geom,
                                                                                                                             filtered_outage_data_gdf['outage_start_time'].iloc[f],
                                                                                                                             filtered_outage_data_gdf['area affected'].iloc[f],
                                                                                                                             sm_data['SME'][sm_data['SME']==sm_data['SME'].max()].values,
                                                                                                                             sm_data.index[sm_data['SME']==sm_data['SME'].max()].values,))
                candidates.append({
                                    'wildfire datetime': wildfire_datetime,
                                    'wildfire geometry': geom,
                                    'outage_start_time': filtered_outage_data_gdf['outage_start_time'].iloc[f],
                                    'outage_stop_time': filtered_outage_data_gdf['outage_stop_time'].iloc[f],
                                    'customers affected': filtered_outage_data_gdf['customers_affected'].iloc[f],
                                    'outage geometry': filtered_outage_data_gdf['geometry'].iloc[f],
                                    'area affected': filtered_outage_data_gdf['area affected'].iloc[f],
                                    'max SME': sm_data['SME'][sm_data['SME']==sm_data['SME'].max()].values,

                                    })
                break
    continue
                
            


In [ ]:
candidates

In [ ]:
gdf_candidates

In [ ]:
gdf_candidates = gpd.GeoDataFrame(candidates)
gdf_candidates = gdf_candidates.set_geometry("outage geometry")


# Save with pickle 
gdf_candidates.to_pickle("/Users/ryanmc/Documents/Conferences/Jack_Eddy_Symposium_2022/dev/candidate_multihazards_data/wildfire_space-weather_2023_candidates.pkl")
# loaded_gdf = gpd.read_pickle("hazard_events.pkl")

# # Save to GeoJSON (need to ensure all geometry types are the same before this works)
# gdf_candidates.to_file("/Users/ryanmc/Documents/Conferences/Jack_Eddy_Symposium_2022/dev/candidate_multihazards_data/wildfire_space-weather_2023_candidates.geojson", driver="GeoJSON")

#### Load EM-DAT data

In [ ]:
directory_emdat = '/Users/ryanmc/Documents/Conferences/Jack_Eddy_Symposium_2022/dev/emdat_data/'
file_emdat = 'public_emdat_2023_US.xlsx'
df_emdat = pd.read_excel(os.path.join(directory_emdat,file_emdat))

In [ ]:
df_emdat.columns.tolist()

In [ ]:
df_emdat[['Start Year', 'Start Month', 'Start Day']]
df_emdat[['End Year', 'End Month', 'End Day']]



In [ ]:
df_emdat['start_datetime'] = df_emdat.apply(
    lambda row: pd.Timestamp(int(row['Start Year']), int(row['Start Month']), int(row['Start Day'])) 
    if pd.notna(row['Start Year']) and pd.notna(row['Start Month']) and pd.notna(row['Start Day'])
    else pd.NaT,
    axis=1
)

df_emdat['end_datetime'] = df_emdat.apply(
    lambda row: pd.Timestamp(int(row['End Year']), int(row['End Month']), int(row['End Day'])) 
    if pd.notna(row['End Year']) and pd.notna(row['End Month']) and pd.notna(row['End Day'])
    else pd.NaT,
    axis=1
)



In [ ]:
gdf_candidates.columns.tolist() #

#### Compare potential events across candidates and database(s)

In [ ]:
for idx,row_emdat in df_emdat.iterrows():
    tmp_diff = (gdf_candidates['outage_start_time'] - row_emdat['start_datetime']).abs()
    result = gdf_candidates[tmp_diff <= pd.Timedelta(days=1)]

    print('overlap of time around {}, \n \t location candidate = {} \n \t  location emdat = {}'.format(row_emdat['start_datetime'],result['area affected'],row_emdat['Location']))
    print('\n\n')


    

Check on


overlap of time around 2023-04-01 00:00:00, 
 	 location candidate = 10    [(Arkansas, None)]
11    [(Illinois, None)]
Name: area affected, dtype: object 
 	  location emdat = Texas, Louisiana, Oklahoma, Kansas, Illinois, Missouri, Nebraska, Washington, Oregon, Montana





In [ ]:
gdf_candidates_multihazard = gdf_candidates[(gdf_candidates['outage_start_time'] >= datetime(2023,3,31)) & (gdf_candidates['outage_start_time'] <= datetime(2023,4,2))]


In [ ]:
# import geopandas as gpd
# import folium
# import time
# from shapely.geometry import Point, Polygon
# from IPython.display import display, clear_output


# Create a Folium map centered over CONUS
conus_map = folium.Map(location=[37.5, -95], zoom_start=5)

for index, row in gdf_candidates_multihazard.iterrows():
    geom = row['wildfire geometry']

    # Check if the geometry is a Point
    if geom.geom_type == 'Point':
        marker_layer = folium.Marker(
            location=[geom.y, geom.x],
            popup="Point Geometry"
        ).add_to(conus_map)
    
    # Check if the geometry is a Polygon
    elif geom.geom_type == 'Polygon':
        poly_layer = folium.Polygon(
            locations=[(y, x) for x, y in geom.exterior.coords],
            color="blue",
            fill=True,
            fill_opacity=0.4,
            popup="Polygon Geometry"
        ).add_to(conus_map)

display(conus_map)


In [ ]:
for index, row in gdf_candidates_multihazard.iterrows():
    print(row)

In [ ]:
row['outage geometry']

TODO
- Compile the concrete information that we know about this event
- Determine what continuous data we have (imagery over time, space weather data (GIC and SuperMAG)
- Can these 'layers' (need the weather and space weather spatiotemporal maps) yield network data and how might I explore its information over time?
- get additional data - Geo-electric field
- ...

In [6]:
import pandas as pd
import geopandas as gpd

# Path to the pickle file
pkl_file_path = '/Users/ryanmc/Documents/Conferences/Jack_Eddy_Symposium_2022/dev/candidate_multihazards_data/wildfire_space-weather_2023_candidates.pkl'

# Load the pickle file using pandas
candidate_events_df = pd.read_pickle(pkl_file_path)

# If the DataFrame contains geospatial data, convert it to a GeoDataFrame
if 'outage geometry' in candidate_events_df.columns:
    candidate_events_gdf = gpd.GeoDataFrame(candidate_events_df, geometry='outage geometry')
    # Filter for events in California
    california_events = candidate_events_gdf[candidate_events_gdf['state'] == 'California']
    print(california_events)
else:
    print("The DataFrame does not contain a 'geometry' column.")

KeyError: 'state'

In [4]:
candidate_events_df

,wildfire datetime,wildfire geometry,outage_start_time,outage_stop_time,customers affected,outage geometry,area affected,max SME
0,2023-01-07 04:10:00,"POLYGON ((-97.5908 27.8072, -96.6931 30.5511, ...",2023-01-07 23:44:00,2023-01-08 02:29:00,9823,"POLYGON ((-94.4578 31.03333, -94.45746 31.0334...","[(Texas, Jasper County), (Texas, Tyler County)...",[234.766418]
1,2023-01-11 15:41:00,"POLYGON ((-82.4514 32.6894, -82.4681 32.9785, ...",2023-01-12 14:00:00,2023-01-13 03:00:00,162000,"POLYGON ((-85.00256 31.00068, -85.00275 31.000...","[(Alabama, None), (Georgia, None)]",[503.498718]
2,2023-01-11 15:42:00,"POLYGON ((-82.2786 29.3265, -82.2924 29.328, -...",2023-01-12 14:00:00,2023-01-13 03:00:00,162000,"POLYGON ((-85.00256 31.00068, -85.00275 31.000...","[(Alabama, None), (Georgia, None)]",[503.498718]
3,2023-01-11 18:53:00,"POLYGON ((-82.2929 29.3187, -91.0701 30.0902, ...",2023-01-12 14:00:00,2023-01-13 03:00:00,162000,"POLYGON ((-85.00256 31.00068, -85.00275 31.000...","[(Alabama, None), (Georgia, None)]",[503.498718]
4,2023-01-11 18:54:00,"POLYGON ((-82.7434 32.9896, -95.0548 33.8018, ...",2023-01-12 14:00:00,2023-01-13 03:00:00,162000,"POLYGON ((-85.00256 31.00068, -85.00275 31.000...","[(Alabama, None), (Georgia, None)]",[503.498718]
5,2023-01-23 17:23:00,"POLYGON ((-96.8342 28.6828, -104.7759 30.8824,...",2023-01-24 15:25:00,2023-01-26 20:20:00,100731,"POLYGON ((-94.97884 29.67673, -94.97918 29.676...","[(Texas, Harris County), (, None)]",[767.988037]
6,2023-01-23 19:12:00,"POLYGON ((-90.837 29.6284, -92.4218 29.6549, -...",2023-01-24 15:25:00,2023-01-26 20:20:00,100731,"POLYGON ((-94.97884 29.67673, -94.97918 29.676...","[(Texas, Harris County), (, None)]",[767.988037]
7,2023-01-24 19:55:00,POINT (-99.9677 30.6221),2023-01-25 03:30:00,2023-01-25 13:00:00,60958,"POLYGON ((-88.85491 30.13553, -88.83936 30.101...","[(Arkansas, None), (Texas, None), (Louisiana, ...",[444.479645]
8,2023-02-20 19:21:00,"POLYGON ((-97.9498 27.3197, -98.3125 27.551, -...",2023-02-21 13:14:00,2023-02-21 14:36:00,2547,"POLYGON ((-81.15781 28.61222, -81.14608 28.612...","[(Florida, Orange County), (, None)]",[521.609131]
9,2023-03-22 20:54:00,"POLYGON ((-114.9734 35.869, -114.9641 35.8799,...",2023-03-23 00:39:00,2023-03-23 00:40:00,11479,"POLYGON ((-115.89692 36.84208, -115.89159 36.8...","[(Nevada, Clark County), (, None)]",[1162.580811]


In [7]:
# Read in a pickle file containing a GeoPandas DataFrame of candidate events
import pandas as pd
import geopandas as gpd

# Path to the pickle file
pkl_file_path = '/Users/ryanmc/Documents/Conferences/Jack_Eddy_Symposium_2022/dev/candidate_multihazards_data/wildfire_space-weather_2023_candidates.pkl'

# Load the pickle file using pandas
candidate_events_df = pd.read_pickle(pkl_file_path)

# If the DataFrame contains geospatial data, convert it to a GeoDataFrame
if 'outage geometry' in candidate_events_df.columns:
    candidate_events_gdf = gpd.GeoDataFrame(candidate_events_df, geometry='outage geometry')
    
    # Filter for events in California based on geometry
    california_events = candidate_events_gdf[candidate_events_gdf['outage geometry'].intersects(gpd.GeoSeries.from_file(gpd.datasets.get_path('naturalearth_lowres'))[gpd.GeoSeries.from_file(gpd.datasets.get_path('naturalearth_lowres'))['name'] == 'California'].unary_union)]
    
    # Display the filtered events
    print(california_events)
    
    # Optionally, save the filtered events to a new file
    california_events.to_file('california_events.geojson', driver='GeoJSON')
else:
    print("The DataFrame does not contain a 'geometry' column.")

AttributeError: The geopandas.dataset has been deprecated and was removed in GeoPandas 1.0. You can get the original 'naturalearth_lowres' data from https://www.naturalearthdata.com/downloads/110m-cultural-vectors/.

In [10]:
# Filter candidate events based on intersection with California using gdf_states
import geopandas as gpd

# Path to the shapefile for states
gdf_states = gpd.read_file('/Users/ryanmc/Documents/Conferences/Jack_Eddy_Symposium_2022/dev/location_data/Census_Bureau_Data/tl_2023_us_state/tl_2023_us_state.shp')

# Ensure the GeoDataFrame is in the same CRS as the candidate events
if candidate_events_gdf.crs is None:
    candidate_events_gdf.set_crs("EPSG:4326", inplace=True)  # Set default CRS if missing

if gdf_states.crs is None:
    gdf_states.set_crs("EPSG:4326", inplace=True)  # Set default CRS if missing

if candidate_events_gdf.crs != gdf_states.crs:
    candidate_events_gdf = candidate_events_gdf.to_crs(gdf_states.crs)

# Filter for California
california_boundary = gdf_states[gdf_states['NAME'] == 'California']
california_events = candidate_events_gdf[candidate_events_gdf['outage geometry'].intersects(california_boundary)]

# Display the filtered events
print(california_events)

# Optionally, save the filtered events to a new file
california_events.to_file('../data/california_events.geojson', driver='GeoJSON')

Empty GeoDataFrame
Columns: [wildfire datetime, wildfire geometry, outage_start_time, outage_stop_time, customers affected, outage geometry, area affected, max SME]
Index: []


/var/folders/s_/nsj8l6gd247cf7v6b5_pnsm00000gp/T/ipykernel_26489/1125324949.py:19: UserWarning: The indices of the left and right GeoSeries' are not equal, and therefore they will be aligned (reordering and/or introducing missing values) before executing the operation. If this alignment is the desired behaviour, you can silence this warning by passing 'align=True'. If you don't want alignment and protect yourself of accidentally aligning, you can pass 'align=False'.
  california_events = candidate_events_gdf[candidate_events_gdf['outage geometry'].intersects(california_boundary)]


In [12]:
california_events = candidate_events_gdf[candidate_events_gdf['outage geometry'].intersects(california_boundary['geometry'])]


/var/folders/s_/nsj8l6gd247cf7v6b5_pnsm00000gp/T/ipykernel_26489/2377627761.py:1: UserWarning: The indices of the left and right GeoSeries' are not equal, and therefore they will be aligned (reordering and/or introducing missing values) before executing the operation. If this alignment is the desired behaviour, you can silence this warning by passing 'align=True'. If you don't want alignment and protect yourself of accidentally aligning, you can pass 'align=False'.
  california_events = candidate_events_gdf[candidate_events_gdf['outage geometry'].intersects(california_boundary['geometry'])]


In [13]:
california_events

,wildfire datetime,wildfire geometry,outage_start_time,outage_stop_time,customers affected,outage geometry,area affected,max SME


In [14]:
# Create a GeoDataFrame with selected states and their geometries
import geopandas as gpd

# List of states to include
selected_states = [
    "Texas", "Oklahoma", "Arkansas", "Kansas", "Missouri", "Iowa", "Minnesota", 
    "Wisconsin", "Illinois", "Indiana", "Mississippi", "Tennessee", "Ohio", "West Virginia"
]

# Path to the shapefile for states
gdf_states = gpd.read_file('/Users/ryanmc/Documents/Conferences/Jack_Eddy_Symposium_2022/dev/location_data/Census_Bureau_Data/tl_2023_us_state/tl_2023_us_state.shp')

# Filter the GeoDataFrame for the selected states
selected_states_gdf = gdf_states[gdf_states['NAME'].isin(selected_states)]

# Save the filtered GeoDataFrame to a new file
output_path = '../data/selected_states_MarchApril2023_event.geojson'
selected_states_gdf.to_file(output_path, driver='GeoJSON')

# Display the filtered GeoDataFrame
print(selected_states_gdf)

   REGION DIVISION STATEFP   STATENS GEOID      GEOIDFQ STUSPS           NAME  \
0       3        5      54  01779805    54  0400000US54     WV  West Virginia   
2       2        3      17  01779784    17  0400000US17     IL       Illinois   
3       2        4      27  00662849    27  0400000US27     MN      Minnesota   
15      2        3      55  01779806    55  0400000US55     WI      Wisconsin   
24      2        3      39  01085497    39  0400000US39     OH           Ohio   
25      3        7      48  01779801    48  0400000US48     TX          Texas   
28      3        7      40  01102857    40  0400000US40     OK       Oklahoma   
29      3        6      47  01325873    47  0400000US47     TN      Tennessee   
43      3        7      05  00068085    05  0400000US05     AR       Arkansas   
44      3        6      28  01779790    28  0400000US28     MS    Mississippi   
45      2        4      29  01779791    29  0400000US29     MO       Missouri   
47      2        4      20  

### Power Grid Regional Subsets

Create side-by-side plots of power grid subsets for specific regions:
- **California**: Key western grid infrastructure
- **Northeast Corridor**: DC to Boston megalopolis

In [ ]:
# Regional Power Grid Subsets: California and Northeast Corridor

import matplotlib.pyplot as plt
import geopandas as gpd

# Check if power grid data is loaded
if 'gdf_powergrid' not in globals():
    print("ERROR: gdf_powergrid not found. Please load power grid data first.")
else:
    # Define bounding boxes for regions
    bbox_ca = (-125, 32, -113, 43)  # California: (west, south, east, north)
    bbox_ne = (-77.5, 38.5, -70, 42.8)  # Northeast corridor: DC to Boston
    
    # Extract subsets using bounding box filtering
    ca_mask = (
        (gdf_powergrid.geometry.bounds['minx'] >= bbox_ca[0]) &
        (gdf_powergrid.geometry.bounds['maxx'] <= bbox_ca[2]) &
        (gdf_powergrid.geometry.bounds['miny'] >= bbox_ca[1]) &
        (gdf_powergrid.geometry.bounds['maxy'] <= bbox_ca[3])
    )
    
    ne_mask = (
        (gdf_powergrid.geometry.bounds['minx'] >= bbox_ne[0]) &
        (gdf_powergrid.geometry.bounds['maxx'] <= bbox_ne[2]) &
        (gdf_powergrid.geometry.bounds['miny'] >= bbox_ne[1]) &
        (gdf_powergrid.geometry.bounds['maxy'] <= bbox_ne[3])
    )
    
    gdf_ca = gdf_powergrid[ca_mask]
    gdf_ne = gdf_powergrid[ne_mask]
    
    print(f"California subset: {len(gdf_ca)} features")
    print(f"Northeast corridor subset: {len(gdf_ne)} features")
    
    # Create side-by-side plots
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 8))
    
    # Plot California
    gdf_ca.plot(ax=ax1, linewidth=0.8, edgecolor='blue', alpha=0.6)
    ax1.set_xlim(bbox_ca[0], bbox_ca[2])
    ax1.set_ylim(bbox_ca[1], bbox_ca[3])
    ax1.set_title('California Power Grid', fontsize=14, fontweight='bold')
    ax1.set_xlabel('Longitude')
    ax1.set_ylabel('Latitude')
    ax1.grid(True, alpha=0.3, linestyle='--')
    ax1.set_aspect('equal')
    
    # Add frame
    for spine in ax1.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(2)
        spine.set_color('black')
    
    # Plot Northeast Corridor
    gdf_ne.plot(ax=ax2, linewidth=0.8, edgecolor='red', alpha=0.6)
    ax2.set_xlim(bbox_ne[0], bbox_ne[2])
    ax2.set_ylim(bbox_ne[1], bbox_ne[3])
    ax2.set_title('Northeast Corridor Power Grid (DC to Boston)', fontsize=14, fontweight='bold')
    ax2.set_xlabel('Longitude')
    ax2.set_ylabel('Latitude')
    ax2.grid(True, alpha=0.3, linestyle='--')
    ax2.set_aspect('equal')
    
    # Add frame
    for spine in ax2.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(2)
        spine.set_color('black')
    
    plt.suptitle('Power Grid Regional Subsets', fontsize=16, fontweight='bold', y=0.98)
    plt.tight_layout()
    plt.show()
    
    print("\\n✓ Regional subset plots created successfully")

### Multi-Panel Movie: Hazards + Network Stress + Outages

Generate an animated movie showing:
- **Top Row**: Three hazard panels (Eh, NX3, FRP)
- **Bottom Row**: Network stress (nodes/edges) and power outages
- **Features**: Colorbars for all panels, visible subplot frames, hourly resolution with zero-fill

In [ ]:
# Multi-Panel Movie with Hazards, Network Stress, and Outages
# This cell requires the following data structures:
# - H_geoE_pivot, H_SWDI_pivot, H_wildfire_pivot (hazard data by tile/time)
# - S_nodes_pivot, S_edges_pivot (network stress by component/time)
# - outage_hourly (customers_out by county/time)
# - tiles_gdf, nodes_gdf, edges_gdf_full, counties_gdf (geometries)

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.animation as animation
from matplotlib.colors import LogNorm, Normalize
from matplotlib.collections import PatchCollection
import numpy as np
import pandas as pd
from datetime import timedelta

# Configuration
output_path = './multihazard_movie_with_stress.mp4'
preview_dir = './preview_frames'

# Verify required data structures exist
required_vars = ['H_geoE_pivot', 'H_SWDI_pivot', 'H_wildfire_pivot', 
                 'S_nodes_pivot', 'S_edges_pivot', 'outage_hourly',
                 'tiles_gdf', 'nodes_gdf', 'edges_gdf_full', 'counties_gdf']

missing_vars = [var for var in required_vars if var not in globals()]
if missing_vars:
    print(f"ERROR: Missing required variables: {missing_vars}")
    print("Please ensure all hazard data, stress data, and geometries are loaded.")
else:
    print("✓ All required data structures found. Proceeding with movie generation...")
    
    # Create complete hourly timeline
    all_times = set()
    if 'H_geoE_pivot' in globals() and not H_geoE_pivot.empty:
        all_times.update(H_geoE_pivot.index)
    if 'H_SWDI_pivot' in globals() and not H_SWDI_pivot.empty:
        all_times.update(H_SWDI_pivot.index)
    if 'H_wildfire_pivot' in globals() and not H_wildfire_pivot.empty:
        all_times.update(H_wildfire_pivot.index)
    if 'S_nodes_pivot' in globals() and not S_nodes_pivot.empty:
        all_times.update(S_nodes_pivot.index)
    if 'outage_hourly' in globals() and not outage_hourly.empty:
        all_times.update(outage_hourly.index)
    
    if not all_times:
        print("ERROR: No time data found in any of the pivot tables")
    else:
        # Generate complete hourly range
        min_time = min(all_times)
        max_time = max(all_times)
        hourly_timeline = pd.date_range(start=min_time, end=max_time, freq='1H')
        
        # Reindex all pivot tables to complete hourly timeline with zero-fill
        H_geoE_hourly = H_geoE_pivot.reindex(hourly_timeline, fill_value=0) if 'H_geoE_pivot' in globals() else pd.DataFrame()
        H_SWDI_hourly = H_SWDI_pivot.reindex(hourly_timeline, fill_value=0) if 'H_SWDI_pivot' in globals() else pd.DataFrame()
        H_wildfire_hourly = H_wildfire_pivot.reindex(hourly_timeline, fill_value=0) if 'H_wildfire_pivot' in globals() else pd.DataFrame()
        S_nodes_hourly = S_nodes_pivot.reindex(hourly_timeline, fill_value=0) if 'S_nodes_pivot' in globals() else pd.DataFrame()
        S_edges_hourly = S_edges_pivot.reindex(hourly_timeline, fill_value=0) if 'S_edges_pivot' in globals() else pd.DataFrame()
        outages_hourly = outage_hourly.reindex(hourly_timeline, fill_value=0) if 'outage_hourly' in globals() else pd.DataFrame()
        
        print(f"Timeline: {min_time} to {max_time}")
        print(f"Total frames: {len(hourly_timeline)}")
        print(f"Hourly resolution with zero-fill complete")
        
        # Prepare geometries for plotting
        tiles_plot = tiles_gdf.to_crs("EPSG:4326") if 'tiles_gdf' in globals() else None
        nodes_plot = nodes_gdf.to_crs("EPSG:4326") if 'nodes_gdf' in globals() else None
        edges_plot = edges_gdf_full.to_crs("EPSG:4326") if 'edges_gdf_full' in globals() else None
        counties_plot = counties_gdf.to_crs("EPSG:4326") if 'counties_gdf' in globals() else None
        
        # Get spatial bounds
        bounds = tiles_plot.total_bounds if tiles_plot is not None else None
        
        # Determine color ranges for consistent scaling
        vmin_eh = H_geoE_hourly.min().min() if not H_geoE_hourly.empty else 0
        vmax_eh = H_geoE_hourly.max().max() if not H_geoE_hourly.empty else 1
        
        vmin_nx3 = H_SWDI_hourly.min().min() if not H_SWDI_hourly.empty else 0
        vmax_nx3 = H_SWDI_hourly.max().max() if not H_SWDI_hourly.empty else 1
        
        vmin_frp = H_wildfire_hourly.min().min() if not H_wildfire_hourly.empty else 0
        vmax_frp = H_wildfire_hourly.max().max() if not H_wildfire_hourly.empty else 1
        
        vmin_nodes = S_nodes_hourly.min().min() if not S_nodes_hourly.empty else 0
        vmax_nodes = S_nodes_hourly.max().max() if not S_nodes_hourly.empty else 1
        
        vmin_edges = S_edges_hourly.min().min() if not S_edges_hourly.empty else 0
        vmax_edges = S_edges_hourly.max().max() if not S_edges_hourly.empty else 1
        
        vmin_outage = outages_hourly.min().min() if not outages_hourly.empty else 0
        vmax_outage = outages_hourly.max().max() if not outages_hourly.empty else 1
        
        # Use log norm for hazards if range is large
        use_log_eh = (vmax_eh / vmin_eh) > 100 if vmin_eh > 0 else False
        use_log_nx3 = (vmax_nx3 / vmin_nx3) > 100 if vmin_nx3 > 0 else False
        use_log_frp = (vmax_frp / vmin_frp) > 100 if vmin_frp > 0 else False
        
        print(f"\\nColor ranges:")
        print(f"  Eh: {vmin_eh:.2e} to {vmax_eh:.2e} {'(log scale)' if use_log_eh else ''}")
        print(f"  NX3: {vmin_nx3:.2e} to {vmax_nx3:.2e} {'(log scale)' if use_log_nx3 else ''}")
        print(f"  FRP: {vmin_frp:.2e} to {vmax_frp:.2e} {'(log scale)' if use_log_frp else ''}")
        print(f"  Node stress: {vmin_nodes:.2e} to {vmax_nodes:.2e}")
        print(f"  Edge stress: {vmin_edges:.2e} to {vmax_edges:.2e}")
        print(f"  Outages: {vmin_outage:.0f} to {vmax_outage:.0f}")
        
        # Initialize figure with gridspec layout
        def init_figure():
            fig = plt.figure(figsize=(20, 12))
            gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.3, wspace=0.3,
                                   height_ratios=[1, 1], width_ratios=[1, 1, 1])
            
            # Top row: 3 hazard panels
            ax_eh = fig.add_subplot(gs[0, 0])
            ax_nx3 = fig.add_subplot(gs[0, 1])
            ax_frp = fig.add_subplot(gs[0, 2])
            
            # Bottom row: 2 panels (stress, outages)
            ax_stress = fig.add_subplot(gs[1, :2])  # Span 2 columns
            ax_outage = fig.add_subplot(gs[1, 2])
            
            axes = {
                'eh': ax_eh, 'nx3': ax_nx3, 'frp': ax_frp,
                'stress': ax_stress, 'outage': ax_outage
            }
            
            # Make all axes have visible frames
            for ax in axes.values():
                ax.spines['top'].set_visible(True)
                ax.spines['right'].set_visible(True)
                ax.spines['bottom'].set_visible(True)
                ax.spines['left'].set_visible(True)
                ax.spines['top'].set_linewidth(2)
                ax.spines['right'].set_linewidth(2)
                ax.spines['bottom'].set_linewidth(2)
                ax.spines['left'].set_linewidth(2)
                ax.spines['top'].set_color('black')
                ax.spines['right'].set_color('black')
                ax.spines['bottom'].set_color('black')
                ax.spines['left'].set_color('black')
                
                if bounds is not None:
                    ax.set_xlim(bounds[0], bounds[2])
                    ax.set_ylim(bounds[1], bounds[3])
                ax.set_aspect('equal')
            
            return fig, axes
        
        # Draw hazard on a panel
        def draw_hazard(ax, tiles, values, title, cmap, norm, cbar_label):
            ax.clear()
            ax.set_title(title, fontsize=14, fontweight='bold', pad=10)
            
            # Make frame visible
            for spine in ax.spines.values():
                spine.set_visible(True)
                spine.set_linewidth(2)
                spine.set_color('black')
            
            if tiles is None or values is None or len(values) == 0:
                ax.text(0.5, 0.5, 'No Data', ha='center', va='center',
                        transform=ax.transAxes, fontsize=16, color='gray')
                return None, None
            
            # Join tiles with values
            tiles_data = tiles.copy()
            tiles_data['value'] = values
            tiles_with_data = tiles_data[tiles_data['value'].notna()]
            
            if len(tiles_with_data) == 0:
                ax.text(0.5, 0.5, 'No Data', ha='center', va='center',
                        transform=ax.transAxes, fontsize=16, color='gray')
                return None, None
            
            # Plot tiles as patch collection
            patches = [poly for poly in tiles_with_data.geometry]
            pc = PatchCollection(patches, cmap=cmap, norm=norm, 
                                edgecolor='none', linewidth=0)
            pc.set_array(tiles_with_data['value'].values)
            collection = ax.add_collection(pc)
            
            # Add colorbar
            cbar = plt.colorbar(collection, ax=ax, fraction=0.046, pad=0.04)
            cbar.set_label(cbar_label, fontsize=10)
            
            if bounds is not None:
                ax.set_xlim(bounds[0], bounds[2])
                ax.set_ylim(bounds[1], bounds[3])
            ax.set_aspect('equal')
            ax.axis('off')
            
            return collection, cbar
        
        # Draw network stress
        def draw_stress(ax, nodes, edges, node_values, edge_values, title):
            ax.clear()
            ax.set_title(title, fontsize=14, fontweight='bold', pad=10)
            
            # Make frame visible
            for spine in ax.spines.values():
                spine.set_visible(True)
                spine.set_linewidth(2)
                spine.set_color('black')
            
            if nodes is None or edges is None:
                ax.text(0.5, 0.5, 'No Data', ha='center', va='center',
                        transform=ax.transAxes, fontsize=16, color='gray')
                return None, None, None, None
            
            # Plot edges
            edges_data = edges.copy()
            edges_data['stress'] = edge_values if edge_values is not None else 0
            edges_with_data = edges_data[edges_data['stress'].notna() & (edges_data['stress'] > 0)]
            
            edge_coll = None
            edge_cbar = None
            if len(edges_with_data) > 0:
                edge_lines = [geom for geom in edges_with_data.geometry]
                edge_norm = Normalize(vmin=vmin_edges, vmax=vmax_edges)
                edge_coll = ax.add_collection(
                    plt.matplotlib.collections.LineCollection(
                        [list(line.coords) for line in edge_lines],
                        cmap='YlOrRd', norm=edge_norm, linewidths=1.5
                    )
                )
                edge_coll.set_array(edges_with_data['stress'].values)
                edge_cbar = plt.colorbar(edge_coll, ax=ax, fraction=0.046, pad=0.04,
                                        label='Edge Stress')
            
            # Plot nodes
            nodes_data = nodes.copy()
            nodes_data['stress'] = node_values if node_values is not None else 0
            nodes_with_data = nodes_data[nodes_data['stress'].notna() & (nodes_data['stress'] > 0)]
            
            node_coll = None
            node_cbar = None
            if len(nodes_with_data) > 0:
                node_norm = Normalize(vmin=vmin_nodes, vmax=vmax_nodes)
                node_coll = ax.scatter(
                    nodes_with_data.geometry.x,
                    nodes_with_data.geometry.y,
                    c=nodes_with_data['stress'].values,
                    cmap='plasma', norm=node_norm,
                    s=20, marker='o', edgecolor='none', alpha=0.8
                )
                node_cbar = plt.colorbar(node_coll, ax=ax, fraction=0.046, pad=0.08,
                                        label='Node Stress')
            
            if bounds is not None:
                ax.set_xlim(bounds[0], bounds[2])
                ax.set_ylim(bounds[1], bounds[3])
            ax.set_aspect('equal')
            ax.axis('off')
            
            return edge_coll, edge_cbar, node_coll, node_cbar
        
        # Draw outages
        def draw_outage(ax, counties, values, title):
            ax.clear()
            ax.set_title(title, fontsize=14, fontweight='bold', pad=10)
            
            # Make frame visible
            for spine in ax.spines.values():
                spine.set_visible(True)
                spine.set_linewidth(2)
                spine.set_color('black')
            
            if counties is None or values is None or len(values) == 0:
                ax.text(0.5, 0.5, 'No Data', ha='center', va='center',
                        transform=ax.transAxes, fontsize=16, color='gray')
                return None, None
            
            # Join counties with outage values
            counties_data = counties.copy()
            counties_data['outage'] = values
            counties_with_data = counties_data[counties_data['outage'].notna() & (counties_data['outage'] > 0)]
            
            if len(counties_with_data) == 0:
                ax.text(0.5, 0.5, 'No Outages', ha='center', va='center',
                        transform=ax.transAxes, fontsize=16, color='green')
                return None, None
            
            # Plot counties
            patches = [poly for poly in counties_with_data.geometry]
            norm = Normalize(vmin=vmin_outage, vmax=vmax_outage)
            pc = PatchCollection(patches, cmap='OrRd', norm=norm,
                                edgecolor='gray', linewidth=0.5, alpha=0.7)
            pc.set_array(counties_with_data['outage'].values)
            collection = ax.add_collection(pc)
            
            # Add colorbar
            cbar = plt.colorbar(collection, ax=ax, fraction=0.046, pad=0.04)
            cbar.set_label('Customers Out', fontsize=10)
            
            if bounds is not None:
                ax.set_xlim(bounds[0], bounds[2])
                ax.set_ylim(bounds[1], bounds[3])
            ax.set_aspect('equal')
            ax.axis('off')
            
            return collection, cbar
        
        # Render a single frame
        def render_frame(fig, axes, timestamp):
            # Get data for this timestamp
            eh_vals = H_geoE_hourly.loc[timestamp] if timestamp in H_geoE_hourly.index else None
            nx3_vals = H_SWDI_hourly.loc[timestamp] if timestamp in H_SWDI_hourly.index else None
            frp_vals = H_wildfire_hourly.loc[timestamp] if timestamp in H_wildfire_hourly.index else None
            node_vals = S_nodes_hourly.loc[timestamp] if timestamp in S_nodes_hourly.index else None
            edge_vals = S_edges_hourly.loc[timestamp] if timestamp in S_edges_hourly.index else None
            outage_vals = outages_hourly.loc[timestamp] if timestamp in outages_hourly.index else None
            
            # Create normalizations
            eh_norm = LogNorm(vmin=max(vmin_eh, 1e-6), vmax=vmax_eh) if use_log_eh else Normalize(vmin=vmin_eh, vmax=vmax_eh)
            nx3_norm = LogNorm(vmin=max(vmin_nx3, 1e-6), vmax=vmax_nx3) if use_log_nx3 else Normalize(vmin=vmin_nx3, vmax=vmax_nx3)
            frp_norm = LogNorm(vmin=max(vmin_frp, 1e-6), vmax=vmax_frp) if use_log_frp else Normalize(vmin=vmin_frp, vmax=vmax_frp)
            
            # Draw all panels
            draw_hazard(axes['eh'], tiles_plot, eh_vals, 'Geoelectric Field (Eh)', 'magma', eh_norm, 'V/km')
            draw_hazard(axes['nx3'], tiles_plot, nx3_vals, 'Severe Weather (NX3)', 'magma', nx3_norm, 'Count')
            draw_hazard(axes['frp'], tiles_plot, frp_vals, 'Wildfire (FRP)', 'magma', frp_norm, 'MW')
            draw_stress(axes['stress'], nodes_plot, edges_plot, node_vals, edge_vals, 'Network Stress')
            draw_outage(axes['outage'], counties_plot, outage_vals, 'Power Outages')
            
            # Add overall timestamp
            fig.suptitle(f'Multi-Hazard Analysis: {timestamp.strftime("%Y-%m-%d %H:%M")}',
                        fontsize=18, fontweight='bold', y=0.98)
        
        # Generate preview frames
        def save_previews(timeline, n_frames=5):
            import os
            os.makedirs(preview_dir, exist_ok=True)
            
            indices = np.linspace(0, len(timeline)-1, n_frames, dtype=int)
            
            print(f"\\nGenerating {n_frames} preview frames...")
            for i, idx in enumerate(indices):
                timestamp = timeline[idx]
                fig, axes = init_figure()
                render_frame(fig, axes, timestamp)
                
                preview_path = f"{preview_dir}/frame_{i:03d}_{timestamp.strftime('%Y%m%d_%H%M')}.png"
                plt.savefig(preview_path, dpi=100, bbox_inches='tight')
                plt.close(fig)
                print(f"  Saved preview {i+1}/{n_frames}: {preview_path}")
            
            print(f"✓ Preview frames saved to {preview_dir}/")
        
        # Save 5 preview frames
        save_previews(hourly_timeline, n_frames=5)

In [ ]:
# Generate full animation
# Run this cell after reviewing the preview frames above

print("\\n" + "="*60)
print("GENERATING FULL ANIMATION")
print("="*60)
print(f"Total frames: {len(hourly_timeline)}")
print(f"Output: {output_path}")
print("This may take several minutes...")
print()

# Initialize animation figure
fig_anim, axes_anim = init_figure()

# Storage for collections to update
collections = {}

def update_frame(frame_idx):
    """Update function for animation"""
    timestamp = hourly_timeline[frame_idx]
    
    # Clear all axes and redraw
    render_frame(fig_anim, axes_anim, timestamp)
    
    # Progress indicator
    if frame_idx % 50 == 0:
        progress = 100 * frame_idx / len(hourly_timeline)
        print(f"  Progress: {progress:.1f}% (frame {frame_idx}/{len(hourly_timeline)})")
    
    return []

# Create animation
anim = animation.FuncAnimation(
    fig_anim,
    update_frame,
    frames=len(hourly_timeline),
    interval=100,  # 100ms between frames = 10 fps
    blit=False,
    repeat=False
)

# Save animation
print("\\nSaving animation to file...")
anim.save(output_path, writer='ffmpeg', fps=10, dpi=100)
plt.close(fig_anim)

print(f"\\n{'='*60}")
print("✓ ANIMATION COMPLETE!")
print(f"{'='*60}")
print(f"Saved to: {output_path}")
print(f"Total frames: {len(hourly_timeline)}")
print(f"Duration: {len(hourly_timeline)/10:.1f} seconds")
print(f"Time range: {hourly_timeline[0]} to {hourly_timeline[-1]}")